## ????????

?????? ?????????????, ????????????, ????????? ?? ???????? IC50 ????????? ???????? ???????. ? ?????? ??????? ??????? ????? ???????????? ????????? ?????? ????????? ???????? ??? ??????? ?????? ??????, ? ????? ????????? ?????? ?? ?????????????????? ? ????????????? ???????????.

In [ ]:
# ??????????? ??????????? ??????????
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ??????? ?????? ? ?????????
df = pd.read_csv('../../data/data.csv')
df.drop(columns=["Unnamed: 0"], inplace=True)

In [ ]:
# ???????? ???????? ??????
df.head(10)

In [ ]:
# ??????? ??????????
target = 'IC50'

# ???????? ???????? ?? ??????? ??????????
X = df.drop(columns=[target, 'CC50', 'SI'], errors='ignore')
IC_median = df[target].median()
y = (df[target] > IC_median).astype(int)

In [ ]:
# ?????????????? ?????????
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ????????? ?????? ?? ????????? ? ???????? ???????
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# ????????? ?????? ???????
print("?????? (IC50 > ???????):", y.value_counts().to_dict())

# ????????????? ???????????? ???????
plt.figure(figsize=(12, 6))
sns.countplot(x=y_train, palette="Set2")
plt.title("????????????? ???????")
plt.xlabel('?????')
plt.ylabel('?????????? ??????????')
plt.grid(axis='y', alpha=0.7)
plt.show()

In [ ]:
# ??????? ??????? ??? ?????? ???????
def evaluate(y_true, y_pred, model_name):
    print(f"\n?????????? ??? ??????: {model_name}")
    print("????????:", accuracy_score(y_true, y_pred))
    print("F1-score:", f1_score(y_true, y_pred))
    print("ROC AUC:", roc_auc_score(y_true, y_pred))
    print("??????? ??????:\n", confusion_matrix(y_true, y_pred))

In [ ]:
# ????????????? ?????????
log_params = {'C': [0.05, 0.5, 1.5, 5]}
log_model = GridSearchCV(LogisticRegression(max_iter=7000, solver='liblinear'), log_params, cv=4, scoring='f1')
log_model.fit(X_train, y_train)
y_pred_log = log_model.best_estimator_.predict(X_test)


# ??????????? ????????????? ???????
probs = log_model.best_estimator_.predict_proba(X_test)[:, 1]

# ???????????? FPR ? TPR
fpr, tpr, thresholds = roc_curve(y_test, probs)

# ?????? ?????????????????? ???????
evaluate(y_test, y_pred_log, "????????????? ????????? (IC50 > ???????)")

# ???????????? ??????? ??????
cm = confusion_matrix(y_test, y_pred_log)

# ????????????? ??????? ??????
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Blues)
plt.title('??????? ?????? ??? ????????????? ????????? (IC50 > ???????)')
plt.grid(False)
plt.show()

???????? ????????? ?????????? (AUC=0.935), ?? ? ???????? ??????????? ?????????????????? ?????? (FP=76), ??? ????? ???? ????????? ? ?????????????? ????????????
????????? ??? ??????? ?????? ??? ??????????????? ???????, ?? ?? ??? ????????? ???????.

In [ ]:
# ????????? ???
rf_params = {'n_estimators': [150, 250, 300], 'max_depth': [None, 15, 25]  }
rf_model = GridSearchCV(RandomForestClassifier(random_state=123), rf_params, cv=4, scoring='f1')
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.best_estimator_.predict(X_test)

# ?????? ?????????????????? ???????
evaluate(y_test, y_pred_rf, "????????? ??? (IC50 > ???????)")

# ??????????? ?????????????? ??????
probs = rf_model.best_estimator_.predict_proba(X_test)[:, 1]

# ???????????? FPR ? TPR
fpr, tpr, thresholds = roc_curve(y_test, probs)

# ???????????? ??????? ??????
cm = confusion_matrix(y_test, y_pred_rf)

# ????????????? ??????? ??????
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Greens)
plt.title('????????? ??? ??? ????????????? ????????? (IC50 > ???????)')
plt.grid(False)
plt.show()

???????? ?????????? ?????????? ?? ???? ???????. ??????????? ?????????? ??????. ???????? ????????????? ? ???????? ?????, ????? ????????????? ? ??????????.

In [ ]:
# ?????? XGBoost
xgb_params = {
    'n_estimators': [150, 250, 350],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.15] 
}
xgb_model = GridSearchCV(
    XGBClassifier(eval_metric='logloss', random_state=123),
    xgb_params,
    cv=4,
    scoring='f1'
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.best_estimator_.predict(X_test)

# ?????? ?????????????????? ???????
evaluate(y_test, y_pred_xgb, "XGBoost (IC50 > ???????)")

# ??????????? ?????????????? ??????
probs = xgb_model.best_estimator_.predict_proba(X_test)[:, 1]

# ???????????? FPR ? TPR
fpr, tpr, thresholds = roc_curve(y_test, probs)

# ???????????? ??????? ??????
cm = confusion_matrix(y_test, y_pred_xgb)

# ????????????? ??????? ??????
plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap=plt.cm.Reds)
plt.title('XGBoost ??? ????????????? ????????? (IC50 > ???????)')
plt.grid(False)
plt.show()

XGBoost ????????????? ??????????, ????????? ?? ????????? ?????, ?? ???? ???????????? ? ??????????? ??????? ????????? ? ???????????????.
????????????? ?????? ??? ????? ? ???????? ???????.

In [ ]:
# ???????? ??? ?????? ??????
log_preds = log_model.best_estimator_.predict(X_test)
rf_preds = rf_model.best_estimator_.predict(X_test)
xgb_preds = xgb_model.best_estimator_.predict(X_test)

# ??????? ?????? ??? ??????????? ?????? ??????
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ???????? ??????? ? ???????? ???????
model_names = ["????????????? ?????????", "????????? ???", "XGBoost"]
predictions = [log_preds, rf_preds, xgb_preds]
colors = ['Blues', 'Greens', 'Reds']

# ???????? ?? ?????? ?????? ? ?????? ??????? ??????
for ax, name, preds, color in zip(axes, model_names, predictions, colors):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['?????????????', '?????????????'])
    disp.plot(ax=ax, cmap=color, values_format='d')
    ax.title.set_text(name)
    
# ????? ????????? ???????
plt.suptitle('????????? ??????? ?? ???????? ??????', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

???????????? ???????? ??????? ????????????? ????????? ???????? ????????? ???????? ??? ?????????????. ??????????? ?????? (Random Forest ? XGBoost) ???????? ???????????? ????????????? ??? ????????????? ??????????.

?????? ????????????? ?????? ???????????????? ? ???? ??????, ????????????? ????????? ??????? ? ?????????????????? ???????, ??????????? ?????? ???????????? ????? ???????????????? ????????.

??? ??????????? ????? ???????? Random Forest, ??? ??? ?????????? ???????? ? ????????????. ??? ??????? ??????? ?????? ??? ?????????? ??????? ???????? XGBoost.